# Material Specification Example

In [ ]:
from math import isclose

from isotopes import H1, H2, Al, B, Cr, Cu, Fe, Isotope, Mg, Mn, O, Si, Ti, Zn

from coremaker.materials import Mixture

## Goal
The goal of these examples is that you will learn how to create material mixtures in different use cases.
Reasons to do so may vary, so we will try to show multiple avenues and compare them.

## Case 1: When you know the isotopic number densities directly
If you already have your mixture composition in terms of number densities in $\frac{1}{cm \cdot barn}$, you can create your mixture objects directly from the class' `__init__` method.

Let's say we have Aluminium-6061 to model. From the [PNNL Compendium, revision 2, page 25](https://www.pnnl.gov/main/publications/external/technical_reports/PNNL-15870Rev2.pdf):

Density: 2.7 g/cc

| Element | Weight Fraction \% | Number Density $\left[\frac{1}{cm \cdot barn}\right]$ |
|---------|--------------------|-------------------------------------------------------|
| Mg      | 1                  | 0.000669                                              |
| Al      | 97.2               | 0.058575                                              |
| Si      | 0.6                | 0.000347                                              |
| Ti      | 8.76e-2            | 0.000030                                              |
| Cr      | 0.195              | 0.000061                                              |
| Mn      | 8.76e-2            | 0.000026                                              |
| Fe      | 0.4088             | 0.000119                                              |
| Cu      | 0.275              | 0.000070                                              |
| Zn      | 0.146              | 0.000036                                              |

We want to create this mixture with these numbers. Let's say we want to do this at room temperature, i.e. 25 degrees celsius.

In [ ]:
mix1 = Mixture(
    {Mg: 66.9e-5,
     Al: 0.058575,
     Si: 34.7e-5,
     Ti: 3e-5,
     Cr: 6.1e-5,
     Mn: 2.6e-5,
     Fe: 11.9e-5,
     Cu: 7e-5,
     Zn: 3.6e-5
    }, temperature=25.0)
mix1

As we can see, the resulting `Mixture` has the expected number densities, at the given temperature. The $S_{\alpha\beta}$ list is empty, because we did not declare that any of these elements should have a thermal correction due to their chemical properties. This is the expected behavior in an alloy such as Aluminium-6061, but if we were to model water, as we will below, we will need those to tell the transport codes to use the correct cross sections given the chemical composition.

Our `Mixture`s can also be expanded to show the density of every isotope for every element:

In [ ]:
mix1.isotopes

In [ ]:
Mixture.expand(mix1).isotopes

## Case 2: When you know the weight fractions
The PNNL compendium also gives us the weight fractions, which we can use instead using a classmethod.

In [ ]:
mix2 = Mixture.by_weight_fraction(
    {Mg: 1e-2,
     Al: 0.972,
     Si: 0.6e-2,
     Ti: 8.76e-4,
     Cr: 0.195e-2,
     Mn: 8.76e-4,
     Fe: 0.4088e-2,
     Cu: 0.275e-2,
     Zn: 0.146e-2
    }, density=2.70, temperature=25.0)
mix2

Notice that these are not the exact same numbers as before. This is because the PNNL compendium gives these as approximates.
We can check to see how big of a difference these are.

Therefore, we expect them not to be considered the same mixture by Python

In [ ]:
assert mix2 != mix1

In [ ]:
def mix_difference(m1: Mixture, m2: Mixture) -> dict[Isotope, float]:
    assert set(m1.isotopes.keys()) == set(m2.isotopes.keys())
    return {iso: 1 - m1[iso]/m2[iso] for iso in m1.isotopes}

In [ ]:
mix_difference(mix1, mix2)

As we can see, the errors are capped at a 1% relative difference, which makes sense because most of these isotopes were given with at most 3 digits.

## Case 3: Alloy definition
It is common, when dealing with alloys, that the main component isn't actually given.
One could solve by hand what the full dictionary of weight fractions are, but we have a quality of life method for that.

In [ ]:
mix3 = Mixture.alloy(
    Al,     
    {Mg: 1e-2,
     Si: 0.6e-2,
     Ti: 8.76e-4,
     Cr: 0.195e-2,
     Mn: 8.76e-4,
     Fe: 0.4088e-2,
     Cu: 0.275e-2,
     Zn: 0.146e-2
    },
    density=2.70,
    temperature=25.)

In this case we expect mix2 and mix3 to be very close. So close, that they should be the same, really.

In [ ]:
assert mix2 == mix3

## Case 4: Use pre-defined mixtures

Some things we already found useful enough that we modeled them, and are available out of the box. Al-6061, used in my research reactors, is one such material. Our implementation adheres to the PNNL compedium, so it should be the same!
The temperature may not be the same, so let's reset that.

In [ ]:
from coremaker.materials.aluminium import al6061

In [ ]:
al6061.temperature = 25
assert mix1 == al6061

## Case 5: Water mixtures

Making water mixtures is slightly more complicated, because the density is a function of the mixture temperature, and not a nice function, at that.
This is why we have a module just for that. We are basing our densities on [AECL 7531](https://inis.iaea.org/records/bgt3d-ng660) for heavy water, and CRNL-1533 for light water. The tables for light water are not available online directly (CRNL-1533 was never published), but are published in the [DRAGON project](https://git.oecd-nea.org/dragon/5.1/-/blob/main/Dragon/src/INFWAT.f).

If you want your density data to be consistent with some other evaluation, please let us know and we'll try to support that too.

In [ ]:
from coremaker.materials.water import make_heavy_water, make_light_water

Making light and heavy water is very easy to do. You just need to know the temperature.

In [ ]:
cold_water = make_light_water(4)  # 4 deg C
hot_water = make_light_water(70)  # 70 deg C

def left_is_denser(mix1, mix2, iso=H1) -> bool:
    return mix1[iso] > mix2[iso]

assert left_is_denser(cold_water, hot_water)
assert isclose(cold_water[H1], 2*cold_water[O], rel_tol=1e-12)

In [ ]:
cold_hwater = make_heavy_water(4)  # 4 deg C
hot_hwater = make_heavy_water(70)  # 70 deg C

assert left_is_denser(cold_hwater, hot_hwater, iso=H2)
assert isclose(cold_hwater[H2], 2*cold_hwater[O], rel_tol=1e-12)

Heavy water is rarely 100% pure, except in simulations. That's why we have a purity option for it.

In [ ]:
impure_hwater = make_heavy_water(80, purity=0.99)
hw_mass_per_molecule = 2*H2.mass + O.mass
lw_mass_per_molecule = 2*H1.mass + O.mass
hw_mass = impure_hwater[H2] / 2 * hw_mass_per_molecule
lw_mass = impure_hwater[H1] / 2 * lw_mass_per_molecule
purity = hw_mass / (hw_mass + lw_mass)
assert isclose(purity, 0.99, rel_tol=1e-12)

In PWRs, it is common to use boric acid in the coolant, so we want to support using things other than water in our water mixtures.
This is also useful for when the water is simply impure due to other impurities in the water.

Technically, when you add boric acid you don't just add boron, you get a bunch of $OH^-$ ions too, which would slightly affect density and number densities, but meh.

In [ ]:
borated_water = make_light_water(300., impurities={B: 1000e-6})
b_mass = borated_water[B] * B.mass
other_mass = borated_water[H1] * H1.mass + (borated_water[H1]/2) * O.mass
mass_ratio = b_mass/(other_mass+b_mass)
assert isclose(mass_ratio, 1000e-6, rel_tol=1e-12), mass_ratio